# Sophie AI — Dataset Generator (Colab)

This notebook generates 40 training images for Sophie's LoRA using **Flux.1-schnell** on Colab's free GPU.

## Instructions:
1. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. Wait ~30-45 min for all images to generate
4. Download the zip file at the end
5. Extract into `datasets/sophie_v1/` in your project

> Uses Flux.1-schnell (Apache 2.0 license, no HuggingFace login needed)

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf pillow

In [ ]:
# Cell 2: Check GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    raise RuntimeError('No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: Define all 20 prompts (from base-prompts.yaml, identity_anchor expanded)

IDENTITY = (
    'a 28 year old woman with wavy light brown hair, green eyes, '
    'light freckles, warm smile, thin gold chain necklace'
)

PROMPTS = {
    # --- CLOSE-UP PORTRAITS ---
    'portrait_neutral': f'sphie, {IDENTITY}, headshot portrait, neutral expression, soft natural window lighting, shallow depth of field, Canon EOS R5, 85mm f/1.4, warm tones, editorial photography',
    'portrait_smile': f'sphie, {IDENTITY}, headshot portrait, genuine warm smile, soft golden hour lighting from the side, bokeh background, Canon EOS R5, 85mm f/1.4, warm color grading',
    'portrait_thinking': f'sphie, {IDENTITY}, looking slightly to the side thoughtfully, holding a pencil near her chin, natural daylight, Canon EOS R5, 85mm f/1.4, warm tones, candid feel',
    'portrait_laughing': f'sphie, {IDENTITY}, laughing naturally, eyes slightly closed, bright natural light, white wall background, Canon EOS R5, 85mm f/1.4, editorial portrait',
    'portrait_glasses': f'sphie, {IDENTITY}, wearing round reading glasses pushed up on her head, hair in loose bun, focused expression, soft studio lighting, Canon EOS R5, 85mm f/1.4',

    # --- HALF-BODY ---
    'studio_sweater': f'sphie, {IDENTITY}, wearing cream oversized knit sweater, standing in bright modern design studio, mood boards on wall behind, holding fabric samples, soft natural lighting from large windows, Canon EOS R5, 50mm f/1.8, interior design studio, warm tones',
    'studio_apron': f'sphie, {IDENTITY}, wearing sage green apron over white t-shirt, hair in loose bun, standing at a work table with paint swatches, bright airy studio space with plants, natural daylight, Canon EOS R5, 50mm f/1.8, editorial photography',
    'studio_linen': f'sphie, {IDENTITY}, wearing white linen button-up shirt, sleeves rolled up, leaning against a desk with design sketches, minimalist bright studio, natural light, Canon EOS R5, 50mm f/1.8',
    'showroom_browsing': f'sphie, {IDENTITY}, wearing denim jacket over simple dress, browsing furniture in a modern showroom, touching a sofa fabric, bright retail lighting, candid shot, Canon EOS R5, 35mm f/2.0',
    'cafe_sketching': f'sphie, {IDENTITY}, wearing cozy cardigan, sitting at a cafe table, sketching in a notebook, coffee beside her, warm afternoon light through window, candid lifestyle photography, Canon EOS R5, 50mm f/1.8, shallow depth of field',

    # --- FULL-BODY / ENVIRONMENTAL ---
    'measuring_wall': f'sphie, {IDENTITY}, wearing cream sweater and jeans, holding a tape measure against a wall in an empty room, natural daylight from windows, renovation setting, Canon EOS R5, 35mm f/2.0, full body shot, editorial',
    'organizing_shelf': f'sphie, {IDENTITY}, wearing white linen shirt, organizing books on a wooden bookshelf, focused and happy, bright cozy living room, natural light, Canon EOS R5, 35mm f/2.0',
    'paint_store': f'sphie, {IDENTITY}, wearing denim jacket, holding up two paint sample cards and comparing them, standing in a hardware store paint aisle, commercial lighting, Canon EOS R5, 50mm f/1.8, candid lifestyle',
    'thrift_store': f'sphie, {IDENTITY}, wearing casual cardigan, examining a vintage wooden chair at a flea market, outdoor natural light, slightly overcast, editorial lifestyle photography, Canon EOS R5, 35mm f/2.0',
    'kitchen_reveal': f'sphie, {IDENTITY}, wearing sage apron, standing proudly in a beautifully renovated modern kitchen, arms slightly open, bright and airy space, natural window light, warm tones, Canon EOS R5, 24mm f/2.8, wide angle, interior photography',

    # --- DIFFERENT ANGLES ---
    'three_quarter_left': f'sphie, {IDENTITY}, three-quarter view from left side, soft natural lighting, neutral background, portrait photography, Canon EOS R5, 85mm f/1.4',
    'three_quarter_right': f'sphie, {IDENTITY}, three-quarter view from right side, soft natural lighting, neutral background, portrait photography, Canon EOS R5, 85mm f/1.4',
    'slight_above': f'sphie, {IDENTITY}, shot from slightly above eye level, looking up at camera with friendly expression, natural daylight, Canon EOS R5, 50mm f/1.8, warm tones',
    'profile_view': f'sphie, {IDENTITY}, profile view looking to the right, soft window backlighting creating rim light on hair, Canon EOS R5, 85mm f/1.4, artistic portrait',
    'from_behind_looking_back': f'sphie, {IDENTITY}, looking back over her shoulder with a smile, standing in a bright room, natural light, Canon EOS R5, 50mm f/1.8, candid editorial photography',
}

print(f'Loaded {len(PROMPTS)} prompts')
print(f'Will generate 2 variants each = {len(PROMPTS) * 2} images')

In [ ]:
# Cell 4: Load Flux.1-schnell model
# schnell = fast (4 steps), Apache 2.0 license, no login needed

from diffusers import FluxPipeline
import torch
import gc

print('Loading Flux.1-schnell... (this takes 3-5 minutes)')

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.float16,
)

# CPU offload keeps only the active component on GPU — essential for T4
pipe.enable_sequential_cpu_offload()

# Enable memory-efficient attention
pipe.enable_attention_slicing()

print('Model loaded and ready!')

In [ ]:
# Cell 5: Generate all images (2 variants per prompt = 40 images)

import os
import time
from IPython.display import display, clear_output

OUTPUT_DIR = '/content/sophie_dataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

total = len(PROMPTS) * 2
generated = 0
start_time = time.time()

for name, prompt in PROMPTS.items():
    for variant in range(1, 3):  # 2 variants per prompt
        filename = f'{name}_v{variant}.png'
        filepath = os.path.join(OUTPUT_DIR, filename)

        # Skip if already generated (for resuming)
        if os.path.exists(filepath):
            generated += 1
            print(f'[{generated}/{total}] SKIP (exists): {filename}')
            continue

        seed = hash(f'{name}_{variant}') % (2**32)
        generator = torch.Generator('cpu').manual_seed(seed)

        try:
            image = pipe(
                prompt=prompt,
                num_inference_steps=4,
                guidance_scale=0.0,  # schnell doesn't use guidance
                height=1024,
                width=1024,
                generator=generator,
            ).images[0]

            image.save(filepath)
            generated += 1

            elapsed = time.time() - start_time
            per_img = elapsed / generated
            remaining = per_img * (total - generated)
            print(f'[{generated}/{total}] Saved: {filename}  '
                  f'({per_img:.0f}s/img, ~{remaining/60:.0f}min left)')

        except Exception as e:
            print(f'[{generated}/{total}] ERROR on {filename}: {e}')
            # Clear VRAM and try to continue
            gc.collect()
            torch.cuda.empty_cache()

total_time = time.time() - start_time
print(f'\nDone! Generated {generated} images in {total_time/60:.1f} minutes')
print(f'Images saved in: {OUTPUT_DIR}')

In [ ]:
# Cell 6: Preview a few images
import matplotlib.pyplot as plt
from PIL import Image
import glob

images = sorted(glob.glob(f'{OUTPUT_DIR}/*.png'))[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path)[:25], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Showing 8 of {len(glob.glob(f"{OUTPUT_DIR}/*.png"))} images')

In [ ]:
# Cell 7: Zip and download
import shutil
from google.colab import files

zip_path = '/content/sophie_dataset'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

zip_file = f'{zip_path}.zip'
size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f'Zip file: {zip_file} ({size_mb:.1f} MB)')
print('Downloading...')
files.download(zip_file)

## Next Steps (on your local machine)

1. Extract `sophie_dataset.zip` into `datasets/sophie_v1/`
2. Review images — delete bad ones (weird hands, wrong face, etc.)
3. Run:
```bash
py scripts/rename_and_pair.py --dir datasets/sophie_v1 --prefix sophie
py scripts/caption_dataset.py --dir datasets/sophie_v1 --trigger sphie --generate-stubs
```
4. Edit each `.txt` caption to describe the actual image
5. Validate:
```bash
py scripts/prepare_dataset.py --dir datasets/sophie_v1
```
6. When it says `VERDICT: Ready for training!` → proceed to RunPod LoRA training